# 🎯 PE Deal Sourcing

*Public private-equity deal announcements, turned into an evidence-backed sourcing record for a deal team.*

**User.** A deal team at a lower-middle-market sponsor sourcing in five sectors: building & facility services, IT
services & cybersecurity, tech-enabled business services, industrial manufacturing and value-added distribution.
**Recurring task.** Every week: which niches are peer sponsors rolling up, which sponsor-owned companies have been held
long enough to be sold soon, and who would we bid against for a target?

This notebook runs the whole workflow:

1. **Setup** — fresh Colab session: clone the project, install dependencies
2. **Collect** — sixteen buy-and-build sponsors' news pages (polite: robots.txt and crawl delays), or the saved pages (default)
3. **Process** — parse, tell deals from other news, extract every field with its evidence, tag sector and niche
4. **Inspect** — the data dictionary, a record next to its source sentences, coverage and failed pages
5. **Validate** — accuracy against hand-labelled pages (held-out vs development), consistency checks, repeatability
6. **Source** — the consolidation map, the exit watch (portfolio ledger), competing buyers, the sourcing brief
7. **Extras** — similar-deal search, the new-deal digest, exports

> **Cached data.** By default everything runs on pages saved by the collector (`data/pe_deals/cache/`, collection time
> in every record), so the notebook gives the same result even if a website changes or goes offline. Set `LIVE = True`
> in step 2 to fetch new pages first (slow by design: up to 10 seconds between pages on some sites).

The same product runs as its own app: `streamlit run app.py` inside `parked/pe-deal-sourcing/` (it is parked, separate
from the Lighthouse stock app). Method, data dictionary and limitations: `docs/PE_DEALS.md`.


## 1 · Setup

In [1]:
# On Google Colab this clones the repository; inside it (or this project's notebooks/ folder) it just finds the package.
import os, sys, subprocess, pathlib
REPO_URL = "https://github.com/Fintech-StThomas/Stock-Trader-FINTECH-Project.git"
PROJECT = pathlib.Path("parked") / "pe-deal-sourcing"          # where the parked project lives in the repository
here = pathlib.Path.cwd()
root = next((p for c in [here, *here.parents] for p in (c, c / PROJECT) if (p / "pedeals" / "__init__.py").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "-q", REPO_URL, "lighthouse_repo"], check=True)
    root = here / "lighthouse_repo" / PROJECT
os.chdir(root); sys.path.insert(0, str(root))
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "requests", "beautifulsoup4", "lxml",
                    "plotly"], check=True, timeout=900)
except Exception as exc:  # offline or already installed - keep going
    print("pip install skipped:", exc)
import pandas as pd
from IPython.display import Markdown, display
import pedeals
from pedeals import build_dataset, analytics as an, charts
from pedeals.sources import SOURCES
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
print("PE Deal Sourcing", pedeals.__version__, "ready in", root)

PE Deal Sourcing 1.0.0 ready in /home/user/Stock-Trader-FINTECH-Project/parked/pe-deal-sourcing


## 2 · Collect

Sixteen buy-and-build sponsors active in the five focus sectors, chosen from a state-by-state research list
(`data/pe_deals/research/pe_firms_by_state.csv`) and kept only if their robots.txt allows their news pages, their news
is listed in a sitemap or listing page, and their articles carry a date. The collector reads each site's robots.txt and obeys it, waits the site's crawl delay
between requests, names itself in the User-Agent, saves every page with its fetch time and logs every URL's outcome.

In [2]:
pd.DataFrame([{"key": k, "firm": s.firm, "home office": s.hq, "sitemaps": ", ".join(s.sitemaps)} for k, s in SOURCES.items()])

,key,firm,home office,sitemaps
0,tonkabay,Tonka Bay Equity Partners,"Minnetonka, MN",https://tonkabayequity.com/wp-sitemap-posts-news-1.xml
1,goldnerhawn,Goldner Hawn,"Minneapolis, MN",https://goldnerhawn.com/wp-sitemap-posts-news-1.xml
2,huron,Huron Capital,"Detroit, MI",https://www.huroncapital.com/news-sitemap.xml
3,align,Align Capital Partners,"Dallas, TX and Cleveland, OH",https://aligncp.com/post-sitemap.xml
4,fortpoint,Fort Point Capital,"Boston, MA",https://fortpointcapital.com/sitemap.xml
5,compass,Compass Group Equity Partners,"St. Louis, MO",https://www.cgep.com/post-sitemap.xml
6,shoreview,ShoreView Industries,"Minneapolis, MN",https://www.shoreview.com/sitemap.xml
7,cid,CID Capital,"Carmel, IN",https://cidcap.com/wp-sitemap-posts-news_release-1.xml
8,masonwells,Mason Wells,"Milwaukee, WI",https://www.masonwells.com/sitemap.xml
9,gryphon,Gryphon Investors,"San Francisco, CA",https://www.gryphon-inv.com/news-sitemap.xml


In [3]:
LIVE = False           # True: fetch new articles from the live sites first (minutes; pages already saved are skipped)
START = "2015-01-01"   # first publication date collected (the ledger needs the older years)

if LIVE:
    from pedeals.collect import collect
    discovery = collect(start=START)
    display(discovery.groupby(["source", "outcome"]).size().rename("urls").to_frame())
else:
    print("Using the saved pages (CACHED DATA). Set LIVE = True to refresh.")

Using the saved pages (CACHED DATA). Set LIVE = True to refresh.


## 3 · Process

`build_dataset` parses every saved page, finds its publication date (page metadata → dateline → the firm's news-page
listing → first date in the text; never guessed), classifies it (deal, personnel, fund / firm, recognition, other),
extracts the deal fields from the release's lead sentence with the evidence for each, infers ownership from the
firm's other announcements where the release is silent, marks duplicates and keeps the period.

In [4]:
ds = build_dataset(start=START)
print(ds.summary_text())

PE Deal Intelligence dataset (CACHED DATA: built from pages saved by `python -m pedeals collect` (fetch time in `collected_at`); the live sites may have changed since.)
period: published 2015-01-01 to 2026-10-29; built 2026-11-12T15:57:01+00:00
1795 articles parsed from 16 firms; 1059 deal announcements in the period, 1048 unique deals, 1 failed pages
publisher                      transaction_type            
Align Capital Partners         Add-on acquisition               86
                               Exit                             12
                               Growth / minority investment      5
                               Merger / combination              3
                               Platform investment              24
                               Unclear                           1
Azalea Capital                 Add-on acquisition                3
                               Exit                              6
                               Platform investment

In [5]:
# Coverage: what each sitemap listed, what was skipped and why, what was kept
ds.coverage

,source,firm,hq,sitemap_urls,skipped_before_period,skipped_robots,pages_parsed,failed_pages,outside_period,articles_in_period,deals_in_period,non_deal_news_in_period,duplicates_marked,earliest_in_period,latest_in_period,collected_at
0,tonkabay,Tonka Bay Equity Partners,"Minnetonka, MN",131,55,0,75,1,10,65,41,24,0,2015-01-06,2026-07-07,2026-11-02T20:56:37+00:00
1,goldnerhawn,Goldner Hawn,"Minneapolis, MN",21,0,0,21,0,0,21,17,4,0,2024-01-05,2026-10-13,2026-10-29T15:03:53+00:00
2,huron,Huron Capital,"Detroit, MI",195,0,0,195,0,43,152,119,33,1,2015-01-23,2026-07-01,2026-11-02T21:05:37+00:00
3,align,Align Capital Partners,"Dallas, TX and Cleveland, OH",171,0,0,171,0,0,171,131,40,0,2016-04-01,2026-10-07,2026-11-02T20:58:43+00:00
4,fortpoint,Fort Point Capital,"Boston, MA",109,0,0,109,0,9,100,72,28,0,2015-03-20,2026-10-22,2026-11-02T21:24:46+00:00
5,compass,Compass Group Equity Partners,"St. Louis, MO",61,0,0,61,0,0,61,31,30,2,2017-06-27,2026-10-05,2026-11-02T21:25:57+00:00
6,shoreview,ShoreView Industries,"Minneapolis, MN",154,0,0,154,0,55,99,74,25,0,2015-03-01,2026-07-15,2026-11-02T21:28:31+00:00
7,cid,CID Capital,"Carmel, IN",73,1,0,72,0,1,71,52,19,0,2015-04-08,2026-10-05,2026-11-02T21:21:41+00:00
8,masonwells,Mason Wells,"Milwaukee, WI",90,0,0,90,0,24,66,50,16,1,2015-01-01,2026-10-14,2026-11-02T21:23:11+00:00
9,gryphon,Gryphon Investors,"San Francisco, CA",335,0,0,335,0,45,290,217,73,3,2015-02-12,2026-10-28,2026-11-02T21:43:36+00:00


In [6]:
# Pages that could not be fetched or parsed (with the reason), and articles with no publication date
display(ds.failures)
ds.records.loc[ds.records["published_date"] == "", ["publisher", "headline", "period_note"]]

,source,url,stage,reason,fetched_at
0,tonkabay,https://tonkabayequity.com/news/tonka-bay-invests-in-quest-events/,fetch,HTTP 504,2026-11-02T20:53:07+00:00


,publisher,headline,period_note
0,Tonka Bay Equity Partners,AccuMold Wins Award,no publication date found
1,Tonka Bay Equity Partners,Corporate Technologies Acquires CCSI,no publication date found
2,Tonka Bay Equity Partners,ESP Announces Acquisition of Weksler Glass Thermometer Corporation (WGTC),no publication date found
3,Tonka Bay Equity Partners,Fenix Specialty Coatings and National Coatings Inc. Join Forces,no publication date found
4,Tonka Bay Equity Partners,NuSource acquires SBM,no publication date found
5,Tonka Bay Equity Partners,P.T. International acquires LMS Bearings,no publication date found
6,Tonka Bay Equity Partners,Tonka Bay Announces Sale of CORWIL Technology Corporation,no publication date found
7,Tonka Bay Equity Partners,Tonka Bay Announces Sale of P.T. International,no publication date found
8,Tonka Bay Equity Partners,Tonka Bay Announces Sale of Q Management,no publication date found
9,Tonka Bay Equity Partners,Vince Thao Named a “Rising Star” by ACG MN,no publication date found


## 4 · Inspect

### Data dictionary
Every output field is defined in `docs/PE_DEALS.md` §5. The categories are defined in code, in one place:

In [7]:
from pedeals.taxonomy import NEWS_CATEGORIES, TRANSACTION_TYPES, DEAL_STATUS, INDUSTRIES, RATIONALE_TAGS
display(pd.DataFrame(TRANSACTION_TYPES.items(), columns=["transaction_type", "definition"]))
display(pd.DataFrame(DEAL_STATUS.items(), columns=["deal_status", "definition"]))
display(pd.DataFrame(NEWS_CATEGORIES.items(), columns=["news_category", "definition"]))
pd.DataFrame([(k, ", ".join(v[:8]) + " ...") for k, v in INDUSTRIES.items()], columns=["target_industry", "example stems"])

,transaction_type,definition
0,Platform investment,the publishing firm itself acquires or invests in a company (a new platform ...
1,Add-on acquisition,a company the firm already owns (a portfolio / platform / partner company) a...
2,Growth / minority investment,"the firm invests without taking control (minority, growth equity, structured..."
3,Merger / combination,"two companies join forces or merge, at least one backed by the firm"
4,Exit,"the firm (or a fund it manages) sells a portfolio company, or a portfolio co..."
5,Dividend recapitalization,a portfolio company refinances to return capital to its owners (no new owner)
6,Unclear,the text describes a transaction but its type cannot be read reliably


,deal_status,definition
0,Completed,"the text says the transaction has happened (has acquired, completed, closed,..."
1,Announced / pending,"signed or agreed but not stated as closed (definitive agreement, will acquir..."
2,Unknown,neither is stated


,news_category,definition
0,Deal,"a transaction: an investment, acquisition, sale, recapitalization or combina..."
1,Personnel,"hires, promotions, appointments, board changes"
2,Fund / firm,"fund closes, new offices, firm milestones, anniversaries"
3,Recognition / media,"awards, rankings, podcasts, interviews, articles, events"
4,Other,"anything else (hiring notices, thought leadership, community news)"


,target_industry,example stems
0,Industrial manufacturing,"manufactur, fabricat, machin, precision, component, bearing, coating, moldin..."
1,Value-added distribution,"distribut, wholesale, supply house, fulfillment, kitting ..."
2,Aerospace & defense,"aerospace, defense, aircraft, aviation, military, space ..."
3,IT services & cybersecurity,"managed service, managed it, it service, it solution, msp, cybersecurity, cy..."
4,Software & data,"software, saas, analytics, data platform, digital platform, application, gis..."
5,Business services,"staffing, consult, outsourc, collection, receivable, screening, background c..."
6,Healthcare,"healthcare, health care, medical, dental, dermatolog, clinic, physician, pat..."
7,Building & infrastructure services,"construction, hvac, plumbing, roofing, electrical contract, electrical servi..."
8,Transportation & logistics,"logistics, trucking, freight, transportation, fleet management, fleet servic..."
9,Consumer & food,"consumer, retail, restaurant, food, beverage, apparel, produce, pet ..."


### The brief's example, as a record

Tonka Bay published the Standard Locknut / Jones Bearing announcement; Tonka Bay is **not** the buyer. Each value
below sits next to the words it was read from.

In [8]:
deals = ds.unique_deals
jones = deals[deals["headline"].str.contains("Jones Bearing")].iloc[0]
Markdown(an.deal_card(jones))

#### Standard Locknut acquires Jones Bearing

| Field | Value | Evidence |
|---|---|---|
| Publisher (PE firm) | Tonka Bay Equity Partners | source page: https://tonkabayequity.com/news/standard-locknut-acquires-jones-bearing/ |
| Published | 2026-07-07 | dateline: MINNETONKA, MN (July 7, 2026) — |
| Transaction type | Add-on acquisition | the buyer is a Tonka Bay Equity Partners company: MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Buyer / investor | Standard Locknut | MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Buyer's relationship to the publisher | Tonka Bay Equity Partners portfolio company | the buyer is a Tonka Bay Equity Partners company: MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Target | Jones Bearing | MINNETONKA, MN (July 7, 2026) — Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Target's business | Based in Alabama, Jones Bearing is a manufacturer of mounted roller bearings and power transmission components used in industrial applications. | announcement text |
| Target industry (inferred) | Industrial manufacturing | stems matched: manufactur, component, bearing |
| Focus sector / niche | Industrial manufacturing / Precision components & machining | niche stems: component, bearing |
| Target's customers / end markets | Industrial / OEM | from 'serving', 'used in', 'customers in' phrases |
| Target location | Alabama (AL) | — |
| Stated rationale | The acquisition expands Standard Locknut’s product offering into light-to-medium-duty mounted bearings while broadening its reach across both OEM and distributor channels. | tags: Product / service expansion; Customers / channels |
| Deal status | Completed | Standard Locknut, a Tonka Bay partner company, has acquired Jones Bearing. |
| Closing date | Not stated | the publication date is not a closing date |
| Price | Not stated | — |
| Ownership | Unknown | — |

*Review flags:* date from dateline

In [9]:
# All deal announcements in the period (the main columns)
deals[an.DISPLAY_COLUMNS].head(15)

,published_date,publisher,transaction_type,buyer,target,target_industry,target_state,rationale_tags,price_disclosure,deal_status,url
0,2015-01-05,Mason Wells,Exit,Unknown,Paris Presents,Unknown,IL,,Not stated,Unknown,https://www.masonwells.com/mason-wells-announces-closing-of-the-sale-of-pari...
1,2015-02-04,Azalea Capital,Exit,"Mosaic Capital Partners, LLC","ETAK Systems, Inc",IT services & cybersecurity,NC,Capacity / scale,Not stated,Unknown,https://azaleacapital.com/azalea-capital-sells-etak-systems-inc/
2,2015-02-09,Mason Wells,Exit,Unknown,Charter NEX Films,Unknown,WI,Capacity / scale,Not stated,Unknown,https://www.masonwells.com/mason-wells-announces-closing-of-the-sale-of-char...
3,2015-02-11,Huron Capital,Add-on acquisition,"Ronnoco Coffee, LLC","U.S. Roasterie, Inc",Industrial manufacturing,IA,Geographic reach; Customers / channels,Not stated,Completed,https://www.huroncapital.com/news/ronnoco-coffee-acquires-u-s-roasterie/
4,2015-03-01,ShoreView Industries,Add-on acquisition,"G.E.T. Enterprises, LLC","Bugambilia International, Inc",Value-added distribution,,Product / service expansion; Geographic reach,Not stated,Completed,https://www.shoreview.com/shoreview-announces-g-e-t-enterprises-llc-has-acqu...
5,2015-03-05,Oridian Capital Partners,Platform investment,Oridian Capital Partners,Tribar Manufacturing,Industrial manufacturing,,Product / service expansion,Not stated,Unknown,https://oridiancapital.com/acquisition-of-tribar-manufacturing/
6,2015-03-09,Tonka Bay Equity Partners,Platform investment,Tonka Bay Equity Partners,Seaway Plastics Engineering,Industrial manufacturing,,Customers / channels; Talent / leadership,Not stated,Completed,https://tonkabayequity.com/news/tonka-bay-invests-in-seaway-plastics-enginee...
7,2015-03-31,Huron Capital,Add-on acquisition,"Albireo Energy, LLC","Sky Technologies, Inc",Energy & environmental,NY,Geographic reach; Technology / capability,Not stated,Completed,https://www.huroncapital.com/news/albireo-energy-acquires-sky-technologies/
8,2015-04-06,Tonka Bay Equity Partners,Platform investment,Tonka Bay Equity Partners,NuSource Financial,Business services,MN,,Not stated,Completed,https://tonkabayequity.com/news/tonka-bay-invests-in-nusource-financial/
9,2015-04-09,CID Capital,Platform investment,CID Capital,"Fit & Fresh, Inc",Consumer & food,RI,,Not stated,Completed,https://cidcap.com/news/cid-capital-partners-with-fit-fresh/


## 5 · Validate

Articles were labelled field by field from the source pages. The rules were improved on a **development** set
(in-sample) and frozen in git before each **held-out** round was drawn and labelled; the newest round is the accuracy
to quote. Consistency checks run on every record; repeatability compares two builds.

**Read this before using the sector views:** deals, dates, buyers and targets hold up on new sponsors, but the sector
and niche matched the labels only about half and a third of the time on the latest round. Treat section 6 as a first
cut and check each company's description.

In [10]:
from pedeals.validate import full_report
Markdown(full_report(ds))

# PE Deal Sourcing: validation

Accuracy per field against articles labelled from the source pages. Each held-out round was labelled after the rules were frozen in git; the newest round measures the rules as they are now and is the number to quote. Rules changed after a round's mismatches were read make that round in-sample, so its score at the freeze is kept next to its score now.

- **held-out-3**: round 3: 44 articles, any sponsor, labelled after the rules were frozen in commit 1835b7f (44 matched)
- **held-out-2**: round 2: 60 articles from the twelve added sponsors and 2015-2022, labelled after the freeze in commit 0bf8a14 (60 matched)
- **held-out**: round 1: 38 articles from the first four sponsors, labelled after the first freeze (38 matched)
- **development**: the articles the first rules were written on (in-sample) (71 matched)

| Field | **held-out-3 (quote this)** | held-out-2 at its freeze | held-out-2 now | held-out now | development now |
|---|---|---|---|---|---|
| is_deal | 42/44 (95%) | 60/60 (100%) | 60/60 (100%) | 38/38 (100%) | 71/71 (100%) |
| news_category | 40/44 (91%) | 58/60 (97%) | 58/60 (97%) | 38/38 (100%) | 71/71 (100%) |
| published_date | 44/44 (100%) | 60/60 (100%) | 60/60 (100%) | 38/38 (100%) | 71/71 (100%) |
| transaction_type | 30/36 (83%) | 43/48 (90%) | 45/48 (94%) | 30/30 (100%) | 54/54 (100%) |
| buyer | 29/36 (81%) | 44/48 (92%) | 45/48 (94%) | 30/30 (100%) | 54/54 (100%) |
| target | 32/36 (89%) | 45/48 (94%) | 45/48 (94%) | 30/30 (100%) | 53/54 (98%) |
| target_industry | 18/36 (50%) | 26/47 (55%) | 39/47 (83%) | 27/30 (90%) | 50/53 (94%) |
| target_state | 25/36 (69%) | 40/48 (83%) | 44/48 (92%) | 29/30 (97%) | 52/54 (96%) |
| deal_status | 27/36 (75%) | 42/48 (88%) | 42/48 (88%) | 29/30 (97%) | 54/54 (100%) |
| price_disclosure | 33/36 (92%) | 46/48 (96%) | 46/48 (96%) | 30/30 (100%) | 54/54 (100%) |
| focus_sector | 18/36 (50%) | 30/47 (64%) | 39/47 (83%) | — | — |
| niche | 8/27 (30%) | 13/31 (42%) | 18/31 (58%) | — | — |

95% Wilson intervals, held-out-3: is_deal 85%-99%; news_category 79%-96%; published_date 92%-100%; transaction_type 68%-92%; buyer 65%-90%; target 75%-96%; target_industry 34%-66%; target_state 53%-82%; deal_status 59%-86%; price_disclosure 78%-97%; focus_sector 34%-66%; niche 16%-48%

Deal vs other news (held-out-3): precision 97%, recall 97% (35 deals found, 1 false alarms, 1 missed).

## Consistency checks (every record, no labels needed)

| Check | Records failing |
|---|---|
| publication date after the collection date | 0 |
| publication date in the future | 0 |
| deal without a source URL | 0 |
| deal without category evidence | 0 |
| publisher recorded as an add-on buyer | 0 |
| platform investment not by the publisher | 0 |
| buyer and target the same | 0 |
| price recorded as zero | 0 |
| price disclosed without a value | 0 |
| closing date before the publication date by > 1 year | 0 |
| duplicate deal ids | 0 |
| platform or add-on with no buyer and no target | 0 |

Repeatability: two builds from the same saved pages are identical (fingerprint `dc53e4d50f3ed47d`).

## Mismatches, held-out-3 (101)

| Article | Field | Label | Program |
|---|---|---|---|
| Huron Capital Completes Equity Investment in WD Lab Grown Diamonds | target_industry | Industrial manufacturing | Healthcare |
| Huron Capital Completes Equity Investment in WD Lab Grown Diamonds | focus_sector | Industrial manufacturing | Outside focus |
| Huron Capital Completes Equity Investment in WD Lab Grown Diamonds | niche | Other: Industrial manufacturing |  |
| Azalea Capital Invests in The Muffin Mam Inc. | target_industry | Consumer & food | Industrial manufacturing |
| Azalea Capital Invests in The Muffin Mam Inc. | deal_status | Completed | Unknown |
| Azalea Capital Invests in The Muffin Mam Inc. | focus_sector | Outside focus | Industrial manufacturing |
| Goldner Hawn & Tonka Bay Announce Sale of Quest Events | target_industry | Business services | Transportation & logistics |
| Goldner Hawn & Tonka Bay Announce Sale of Quest Events | focus_sector | Tech-enabled business services | Outside focus |
| Goldner Hawn & Tonka Bay Announce Sale of Quest Events | niche | Event & audiovisual services |  |
| Fort Point Moves up with 3Phase Elevator | is_deal | yes | False |
| Fort Point Moves up with 3Phase Elevator | news_category | Deal | Personnel |
| Fort Point Moves up with 3Phase Elevator | transaction_type | Platform investment |  |
| Fort Point Moves up with 3Phase Elevator | buyer | Fort Point Capital |  |
| Fort Point Moves up with 3Phase Elevator | target | 3Phase Elevator |  |
| Fort Point Moves up with 3Phase Elevator | target_industry | Building & infrastructure services |  |
| Fort Point Moves up with 3Phase Elevator | target_state | MA |  |
| Fort Point Moves up with 3Phase Elevator | deal_status | Completed |  |
| Fort Point Moves up with 3Phase Elevator | price_disclosure | Not stated |  |
| Fort Point Moves up with 3Phase Elevator | focus_sector | Building & facility services |  |
| Fort Point Moves up with 3Phase Elevator | niche | Elevator services |  |
| NuSource acquires Superior Safe & Security | target_industry | Building & infrastructure services | Business services |
| NuSource acquires Superior Safe & Security | focus_sector | Building & facility services | Tech-enabled business services |
| NuSource acquires Superior Safe & Security | niche | Facility services | Investigations & risk services |
| Gridiron Capital Partners with Health Monitor Network | target_industry | Business services | Healthcare |
| Gridiron Capital Partners with Health Monitor Network | price_disclosure | Not stated | Not disclosed (stated) |
| Gridiron Capital Partners with Health Monitor Network | focus_sector | Tech-enabled business services | Outside focus |
| Gridiron Capital Partners with Health Monitor Network | niche | Marketing & communications services |  |
| NRG Controls Announces Strategic Acquisition of Integrated Systems, In | target_state | GA |  |
| NRG Controls Announces Strategic Acquisition of Integrated Systems, In | deal_status | Completed | Unknown |
| Team Drive-Away Establishes Growth Focused Executive Team | news_category | Personnel | Fund / firm |
| Mason Wells Announces Sale of L.B. White | buyer | Modine Manufacturing Company | Unknown |
| Mason Wells Announces Sale of L.B. White | deal_status | Completed | Unknown |
| Montage Partners Acquires Southwest Data Products | niche | Electronics & electrical equipment | Other: Industrial manufacturing |
| Compass Group Invests in Critical Infrastructure Distributor | niche | Seals & industrial MRO | Other: Value-added distribution |
| Cleartelligence Accelerates Growth with Recruitment of Owen Frivold as | target_industry | IT services & cybersecurity | Business services |
| Cleartelligence Accelerates Growth with Recruitment of Owen Frivold as | focus_sector | IT services & cybersecurity | Tech-enabled business services |
| Cleartelligence Accelerates Growth with Recruitment of Owen Frivold as | niche | Other: IT services & cybersecurity | Engineering & consulting |
| Huron Capital’s High Street Insurance Partners Expands Northeast Prese | target_state | none | PA |
| Resilience Capital Partners Acquires Firequip | transaction_type | Platform investment | Add-on acquisition |
| Resilience Capital Partners Acquires Firequip | niche | Seals & industrial MRO | Other: Value-added distribution |
| SEAM Group Expands Services with Acquisition of EESCO | target_industry | Business services | Energy & environmental |
| SEAM Group Expands Services with Acquisition of EESCO | deal_status | Completed | Unknown |
| SEAM Group Expands Services with Acquisition of EESCO | focus_sector | Tech-enabled business services | Outside focus |
| SEAM Group Expands Services with Acquisition of EESCO | niche | Testing, inspection & calibration |  |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | transaction_type | Platform investment | Add-on acquisition |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | target | National Label (assets) | Assets of Global Package Printing Leader National Label |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | target_industry | Industrial manufacturing | Business services |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | target_state | PA |  |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | deal_status | Completed | Unknown |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | price_disclosure | Not stated | Not disclosed (stated) |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | focus_sector | Industrial manufacturing | Tech-enabled business services |
| Lux Global Label Company, Backed by Resilience Capital Partners, Acqui | niche | Packaging & plastics | Other: Tech-enabled business services |
| Gryphon Portfolio Company CORA Health Services Moves into Tennessee, K | buyer | CORA Health Services | Tennessee and Kentucky |
| Gryphon Portfolio Company CORA Health Services Moves into Tennessee, K | target | Champion Physical Therapy; Professional Therapies of Roanoke | Champion Physical Therapy |
| Gryphon Portfolio Company CORA Health Services Moves into Tennessee, K | deal_status | Completed | Unknown |
| Gryphon Investors Announces Close of Gryphon IV at over $1 Billion in  | is_deal | no | True |
| Gryphon Investors Announces Close of Gryphon IV at over $1 Billion in  | news_category | Fund / firm | Deal |
| Mason Wells Acquires Onyx Brands | target_industry | Consumer & food | Industrial manufacturing |
| Mason Wells Acquires Onyx Brands | target_state | AR |  |
| Mason Wells Acquires Onyx Brands | focus_sector | Outside focus | Industrial manufacturing |
| Pike Street Capital Announces Investment in DigiStor, Leading Provider | target_industry | Industrial manufacturing | Software & data |
| Pike Street Capital Announces Investment in DigiStor, Leading Provider | focus_sector | Industrial manufacturing | Outside focus |
| Pike Street Capital Announces Investment in DigiStor, Leading Provider | niche | Electronics & electrical equipment |  |
| AQUA Dermatology Announces Partnership with First Coast Mohs | target_state | FL |  |
| Huron Capital’s Aftermarket Platform Announces Acquisition of Rago Fab | transaction_type | Add-on acquisition | Platform investment |
| Huron Capital’s Aftermarket Platform Announces Acquisition of Rago Fab | buyer | Horsepower (Addictive Desert Designs and DV8 Offroad) | Huron Capital |
| Huron Capital’s Aftermarket Platform Announces Acquisition of Rago Fab | target_industry | Consumer & food | Industrial manufacturing |
| Huron Capital’s Aftermarket Platform Announces Acquisition of Rago Fab | focus_sector | Outside focus | Industrial manufacturing |
| Shermco Industries Acquires KTR Associates, a Power System Engineering | target_industry | Business services | Energy & environmental |
| Shermco Industries Acquires KTR Associates, a Power System Engineering | focus_sector | Tech-enabled business services | Outside focus |
| Shermco Industries Acquires KTR Associates, a Power System Engineering | niche | Engineering & consulting |  |
| WilliamsMarston Acquires California Corporate Tax Consultancy, Moffett | niche | Tax & accounting services | Engineering & consulting |
| Huron Capital Acquires ExperiGreen and Launches North American Residen | target_industry | Building & infrastructure services | Aerospace & defense |
| Huron Capital Acquires ExperiGreen and Launches North American Residen | focus_sector | Building & facility services | Outside focus |
| Huron Capital Acquires ExperiGreen and Launches North American Residen | niche | Landscaping & grounds |  |
| LEARN Behavioral Partners with Indiana-Based Behavior Analysis Center  | target_state | IN |  |
| Jensen Hughes Names Marc Kaplan New President & Chief Operating Office | news_category | Personnel | Other |
| Compass Electronics Group Acquires Protech Global Solutions | target_state | TX |  |
| Resilience Capital Partners further expands UK Print Operations | target_industry | Business services | Unknown |
| Resilience Capital Partners further expands UK Print Operations | focus_sector | Tech-enabled business services | Outside focus |
| Resilience Capital Partners further expands UK Print Operations | niche | Marketing & communications services |  |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | transaction_type | Growth / minority investment | Add-on acquisition |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | buyer | Huron Capital; Salt Creek Capital | Flex Equity Makes Non-Control |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | target | Pacific Shoring | Construction Safety Company |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | target_industry | Industrial manufacturing | Building & infrastructure services |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | target_state | CA |  |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | deal_status | Completed | Unknown |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | focus_sector | Industrial manufacturing | Building & facility services |
| Huron Capital and Salt Creek Capital Announce Investment in Pacific Sh | niche | Other: Industrial manufacturing | Other: Building & facility services |
| Huron Capital’s InterVision Announces Sixth Add-on Acquisition | transaction_type | Add-on acquisition | Unclear |
| Huron Capital’s InterVision Announces Sixth Add-on Acquisition | buyer | InterVision | Unknown |
| Huron Capital’s InterVision Announces Sixth Add-on Acquisition | target_industry | Software & data | Industrial manufacturing |
| Huron Capital’s InterVision Announces Sixth Add-on Acquisition | target_state | CA |  |
| Huron Capital’s InterVision Announces Sixth Add-on Acquisition | focus_sector | Outside focus | Industrial manufacturing |
| Oridian Capital Partners-Backed ACR Acquires RediBagUSA | buyer | ACR (AmerCareRoyal, LLC) | ACR |
| Oridian Capital Partners-Backed ACR Acquires RediBagUSA | target_industry | Value-added distribution | Consumer & food |
| Oridian Capital Partners-Backed ACR Acquires RediBagUSA | target_state | PA |  |
| Oridian Capital Partners-Backed ACR Acquires RediBagUSA | deal_status | Completed | Unknown |
| Oridian Capital Partners-Backed ACR Acquires RediBagUSA | focus_sector | Value-added distribution | Outside focus |
| Oridian Capital Partners-Backed ACR Acquires RediBagUSA | niche | Other: Value-added distribution |  |
| WilliamsMarston Acquires Valuation Consultant Oracle Capital | niche | Tax & accounting services | Engineering & consulting |

## Mismatches, held-out-2 (52)

| Article | Field | Label | Program |
|---|---|---|---|
| Gryphon Investors’ Mechanix Wear Acquires Chicago Protective Apparel | niche | Safety, marking & signage products | Other: Industrial manufacturing |
| ACP Completes Growth Capital Investment in WilliamsMarston LLC | niche | Tax & accounting services | Engineering & consulting |
| ShoreView Announces Sale of G.E.T. Enterprises, LLC | target_industry | Consumer & food | Value-added distribution |
| ShoreView Announces Sale of G.E.T. Enterprises, LLC | deal_status | Completed | Unknown |
| ShoreView Announces Sale of G.E.T. Enterprises, LLC | focus_sector | Outside focus | Value-added distribution |
| CID Capital Expands Investment Team with Addition of Tom Shaw | news_category | Personnel | Other |
| Meazure Learning Announces Acquisition Of Scantron’s Certification And | target | Scantron Certification and Licensure business | Scantron Corporation’s Certification and Licensure |
| Meazure Learning Announces Acquisition Of Scantron’s Certification And | niche | Other: Tech-enabled business services | Testing, inspection & calibration |
| Gryphon Investors Makes Majority Investment in PestRoutes | price_disclosure | Not stated | Not disclosed (stated) |
| XLerate Group Completes Fifth Add-on Deal, Expands into Pennsylvania | target_industry | Business services | Unknown |
| XLerate Group Completes Fifth Add-on Deal, Expands into Pennsylvania | target_state | PA |  |
| XLerate Group Completes Fifth Add-on Deal, Expands into Pennsylvania | focus_sector | Tech-enabled business services | Outside focus |
| XLerate Group Completes Fifth Add-on Deal, Expands into Pennsylvania | niche | Other: Tech-enabled business services |  |
| Huron Capital Launches Sixteenth ExecFactor ® Platform and Completes I | transaction_type | Platform investment | Add-on acquisition |
| Huron Capital Launches Sixteenth ExecFactor ® Platform and Completes I | buyer | Huron Capital | High Street Partners |
| Huron Capital Launches Sixteenth ExecFactor ® Platform and Completes I | target | High Street Partners; Peterson McGregor and Associates | Peterson McGregor and Associates |
| Huron Capital Launches Sixteenth ExecFactor ® Platform and Completes I | deal_status | Completed | Unknown |
| Mason Wells Acquires The Cleveland Electric Laboratories Company | niche | Electronics & electrical equipment | Other: Industrial manufacturing |
| Fort Point Capital Invests in CellSite Solutions | niche | Other: Building & facility services | Telecom & utility infrastructure |
| MGS Acquires European-Based Medical Injection Molder Knudsen Plast A/S | deal_status | Completed | Unknown |
| MGS Acquires European-Based Medical Injection Molder Knudsen Plast A/S | niche | Packaging & plastics | Other: Industrial manufacturing |
| ShoreView Invests in Hamilton Engineering, Inc. | target_industry | Business services | Unknown |
| ShoreView Invests in Hamilton Engineering, Inc. | focus_sector | Tech-enabled business services | Outside focus |
| ShoreView Invests in Hamilton Engineering, Inc. | niche | Engineering & consulting |  |
| Montage Partners Announces Investment in Military Fasteners | target_state | FL | AZ |
| Caylent Acquires Trek10 to Create the Most Comprehensive Dedicated AWS | niche | Managed IT services (MSP) | Other: IT services & cybersecurity |
| Provider of Real Estate and Employee Relocation Services Expands its P | transaction_type | Add-on acquisition | Platform investment |
| Provider of Real Estate and Employee Relocation Services Expands its P | buyer | Provider of real estate and employee relocation services (Sh | ShoreView Industries |
| Provider of Real Estate and Employee Relocation Services Expands its P | target_industry | Business services | Unknown |
| Provider of Real Estate and Employee Relocation Services Expands its P | focus_sector | Tech-enabled business services | Outside focus |
| Provider of Real Estate and Employee Relocation Services Expands its P | niche | Other: Tech-enabled business services |  |
| Gridiron Capital-Backed GSM Outdoors Acquires Gary Yamamoto Custom Bai | buyer | GSM Outdoors | Good Sportsman Marketing Outdoors |
| Gridiron Capital-Backed GSM Outdoors Acquires Gary Yamamoto Custom Bai | target_industry | Consumer & food | Industrial manufacturing |
| Gridiron Capital-Backed GSM Outdoors Acquires Gary Yamamoto Custom Bai | focus_sector | Outside focus | Industrial manufacturing |
| Burton Plumbing Services Completes Third Add-on Acquisition in 2021 | target_state | WI |  |
| Smile Brands and DecisionOne Dental Announce Strategic Partnership | transaction_type | Growth / minority investment | Add-on acquisition |
| Pike Street Names Sarah Karangelen as Finance and Operations Manager | news_category | Personnel | Other |
| Montage Partners Acquires Indo-European Foods, Inc. | target_industry | Consumer & food | Value-added distribution |
| Montage Partners Acquires Indo-European Foods, Inc. | focus_sector | Outside focus | Value-added distribution |
| Huron Capital’s Pueblo Mechanical & Controls Completes Second and Thir | price_disclosure | Not stated | Not disclosed (stated) |
| Tonka Bay Announces Sale of P.T. International | target_industry | Value-added distribution | Industrial manufacturing |
| Tonka Bay Announces Sale of P.T. International | focus_sector | Value-added distribution | Industrial manufacturing |
| Tonka Bay Announces Sale of P.T. International | niche | Seals & industrial MRO | Precision components & machining |
| ShoreView Announces Sale of Milamar Coatings, LLC | deal_status | Completed | Unknown |
| Quarterhill Announces Agreement to Acquire ETC | target | Electronic Transaction Consultants, LLC | ETC |
| Quarterhill Announces Agreement to Acquire ETC | target_state | TX |  |
| Align Capital Partners Realizes Investment in WilliamsMarston | deal_status | Completed | Unknown |
| Gridiron Capital Announces Sale of Engage2Excel | deal_status | Completed | Unknown |
| ShoreView Launches Test and Measurement Investment Strategy, Acquiring | niche | Electronics & electrical equipment | Precision components & machining |
| Jones Lake Management Acquires Water & Wetland | target_industry | Energy & environmental | Building & infrastructure services |
| Jones Lake Management Acquires Water & Wetland | focus_sector | Outside focus | Building & facility services |
| Gryphon Investors Sells MSD Ignition | niche | Electronics & electrical equipment | Precision components & machining |

## Mismatches, held-out (5)

| Article | Field | Label | Program |
|---|---|---|---|
| Align Partners with Engineering Services Firm AKS | target_state | OR |  |
| Counsel Press Expands into the Trial Courts with Case Anywhere Acquisi | target_industry | Software & data | IT services & cybersecurity |
| Marco Sealing Solutions Continues Expansion with Acquisition of Pride  | target_industry | Value-added distribution | Industrial manufacturing |
| Schneider Geospatial Expands Florida Presence | deal_status | Unknown | Completed |
| Huron Capital’s Platform 1-800 HANSONS Acquires Paramount Builders, LL | target_industry | Building & infrastructure services | Consumer & food |

## Notes

- Every labelled URL is in the dataset.
- Labels were made by reading each source page; see data/pe_deals/validation/README.md for who labelled what and how. The intervals are 95% Wilson intervals: with 30-50 labelled deals they are wide.


## 6 · Source

### Where is the consolidation? (add-ons by niche of the five focus sectors)
Momentum compares the last twelve months of add-ons with the twelve before.

In [11]:
from pedeals import sourcing as so
since_2022 = deals[deals["published_date"] >= "2022-01-01"]
cons = so.consolidation(since_2022)
display(cons.head(15))
charts.horizontal_counts(cons.head(15), "niche", "add_ons", "Add-ons by niche since 2022", "add-ons").show()

,focus_sector,niche,add_ons,platforms,sponsors,last_12m,prior_12m,latest,target_states,momentum
0,Building & facility services,Landscaping & grounds,22,5,3,1,11,2025-11-19,"OH (4), IN (3), WI (1), MA (1), PA (1)",-10
1,Industrial manufacturing,Other: Industrial manufacturing,16,14,6,2,3,2026-03-16,"CA (2), NH (1), PA (1), IL (1), KY (1)",-1
2,Tech-enabled business services,Engineering & consulting,15,9,4,2,2,2025-12-09,"CA (2), SC (1), AZ (1), GA (1), WA (1)",0
3,Building & facility services,HVAC & mechanical,15,8,3,1,2,2026-06-01,"OH (2), UT (1), NM (1), CO (1), MS (1)",-1
4,Value-added distribution,Other: Value-added distribution,12,7,5,2,2,2026-10-06,"MS (1), NC (1), OH (1), KY (1), TX (1)",0
5,IT services & cybersecurity,Managed IT services (MSP),11,4,4,2,3,2026-04-30,"MN (2), ND (1), IA (1), IL (1), UT (1)",-1
6,Building & facility services,Roofing & building envelope,8,4,4,2,1,2026-05-05,"CA (2), TX (2), GA (1), KY (1), MS (1)",1
7,Industrial manufacturing,Precision components & machining,7,5,4,5,2,2026-07-08,"CA (2), NY (1), OR (1), AL (1)",3
8,Tech-enabled business services,"Testing, inspection & calibration",6,4,3,1,2,2026-03-17,"NC (2), IL (1), NJ (1), OH (1)",-1
9,Value-added distribution,Seals & industrial MRO,6,1,1,2,1,2026-09-03,"NV (1), CA (1), CO (1), TX (1)",1


### Exit watch: sponsor-owned companies held long enough to be sold soon

The portfolio ledger rebuilds each sponsor's holdings from its own announcements: an investment announcement or a
sentence that dates it ("ACP acquired StenTech in May of 2022") opens a holding, add-ons show it is still owned, a
sale closes it, a rename ("rebranded as CompletePet") links the names. Lower-middle-market sponsors usually sell after
four to six years.

In [12]:
led = so.ledger(ds.records)
exited = led[(led["status"] == "Exited") & (led["years_basis"] == "from entry")]["years_held"]
print(f"{len(led)} holdings found; {len(exited)} exits with a dated entry, median hold {exited.median():.1f} years")
watch = so.exit_watch(led, min_years=4, sectors=so.FOCUS_SECTORS)
display(watch[["company", "sponsor", "focus_sector", "niche", "years_held", "entry_date", "entry_basis", "add_ons", "signal"]])
charts.years_held(watch).show()

537 holdings found; 85 exits with a dated entry, median hold 5.6 years


,company,sponsor,focus_sector,niche,years_held,entry_date,entry_basis,add_ons,signal
0,E Source,Align Capital Partners,Tech-enabled business services,Engineering & consulting,7.449692,2019-06-01,date stated in a release,14,strong
1,Mechanix Wear®,Gryphon Investors,Industrial manufacturing,Other: Industrial manufacturing,7.233402,2019-08-19,investment announced,3,strong
2,Ncontracts,Gryphon Investors,Tech-enabled business services,Investigations & risk services,6.803559,2020-01-23,investment announced,4,strong
3,Marco Sealing Solutions,Align Capital Partners,Value-added distribution,Seals & industrial MRO,6.778919,2020-02-01,date stated in a release,10,strong
4,KGM,Compass Group Equity Partners,Value-added distribution,Other: Value-added distribution,5.861739,2021-01-01,date stated in a release,1,strong
5,PWSG,CID Capital,Value-added distribution,Lighting & electrical supplies,5.861739,2021-01-01,date stated in a release,1,strong
6,Right Time Group of Companies,Gryphon Investors,Building & facility services,HVAC & mechanical,5.675565,,not found; first seen 2021-03-10,12,watch (entry not dated)
7,Professional,Tonka Bay Equity Partners,Tech-enabled business services,Receivables & collections,5.297741,2021-07-26,investment announced,3,strong
8,IDENTCO,Mason Wells,Industrial manufacturing,Other: Industrial manufacturing,5.196441,2021-09-01,date stated in a release,0,strong
9,PumpTech LLC,Pike Street Capital Partners,Value-added distribution,Other: Value-added distribution,5.163587,2021-09-13,investment announced,2,strong


### Competing buyers and the sourcing brief

*"We want an MSP. Who else is buying them, what did they buy, and which sponsor-owned MSPs may come to market?"*
Change `SECTOR` and `NICHE`.

In [13]:
SECTOR, NICHE = "IT services & cybersecurity", "Managed IT services (MSP)"
display(so.competing_buyers(since_2022, led, [NICHE], [SECTOR]))
Markdown(so.sourcing_brief(since_2022, led, SECTOR, NICHE))

,platform,sponsor,add_ons_here,last_add_on,target_states,niches,platform_status,years_held
0,Cyber Advisors,Goldner Hawn,5,2025-02-27,"IL, MN, ND, UT",Managed IT services (MSP),Exited,2.094456
1,Corporate Technologies,Tonka Bay Equity Partners,4,2026-04-30,"IA, IN",Managed IT services (MSP),No exit announced,3.863107
2,ISPN,Align Capital Partners,1,2026-03-10,GA,Managed IT services (MSP),No exit announced,1.946612


### Sourcing brief: IT services & cybersecurity / Managed IT services (MSP)

**Who is consolidating it** (3 platforms, 10 add-ons in the collected announcements):

- **Cyber Advisors** (Goldner Hawn): 5 add-on(s), latest 2025-02-27; buys in IL, MN, ND, UT
- **Corporate Technologies** (Tonka Bay Equity Partners): 4 add-on(s), latest 2026-04-30; buys in IA, IN
- **ISPN** (Align Capital Partners): 1 add-on(s), latest 2026-03-10; buys in GA

**What they bought** (the target profile to look for):

- 2026-04-30 · **RPM Technologies, LLC** (IN) → Corporate Technologies: Based in Indianapolis, RPM is a national provider of IT services to small and medium-sized business customers, with a focus within the life sciences sector. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-rpm/)
- 2026-03-10 · **ZCorum** (GA) → ISPN: Headquartered in Alpharetta, Georgia, ZCorum provides a suite of broadband diagnostics and managed services designed primarily to help rural internet providers increase operational… [source](https://aligncp.com/news/ispn-and-zcorum-unite-to-expand-managed-services-for-broadband-providers/)
- 2025-09-03 · **Cenetric** → Corporate Technologies: Based in Kansas City, Cenetric is a regional provider of IT managed services focused on small-to-medium sized businesses. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-cenetric/)
- 2025-05-29 · **Travel Nurse Across America; Stella Point; TotalMed** → Unknown: In connection with the transaction, TNAA’s MSP, SimpliFi, will be combining with TotalMed’s MSP, Staffency, bringing together two innovative, trusted providers of tech-enabled work… [source](https://gridironcapital.com/gridiron-capital-and-stella-point-capital-announce-the-strategic-merger-of-portfolio-companies-tnaa-and-totalmed/)
- 2025-02-27 · **Wasatch I.T** (UT) → Cyber Advisors: a prominent IT services company based in Utah [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-managed-service-provider-wasatch-i-t/)
- 2024-11-06 · **eDot** (IL) → Cyber Advisors: a premier Managed Service Provider dedicated to serving small and medium-sized businesses [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-managed-service-provider-edot/)
- 2024-10-16 · **RSPN** (IA) → Corporate Technologies: Based in Cedar Falls, Iowa, RSPN is a regional provider of IT managed services, hosted services, and project support focused on small-to-medium sized businesses. [source](https://tonkabayequity.com/news/corporate-technologies-acquires-rspn/)
- 2024-06-03 · **Total Networx** (MN) → Cyber Advisors: a cybersecurity-focused managed IT services company providing comprehensive, responsive, and cost-effective IT solutions for small to large businesses [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-acquires-total-networx/)
- 2024-05-09 · **S&L Computer Services** (ND) → Cyber Advisors: a managed IT service provider for small and medium-sized businesses [source](https://goldnerhawn.com/news/goldner-hawn-portfolio-company-cyber-advisors-completes-acquisition-of-sl-computer-services/)
- 2024-02-13 · **Innovative Technology Partners** (MN) → Cyber Advisors: a focused provider of personalized managed IT services [source](https://goldnerhawn.com/news/cyber-advisors-completes-acquisition-of-innovative-technology-partners/)
- 2024-01-30 · **NuMSP** → Corporate Technologies: another national MSP that offers IT solutions for small and mid-sized businesses [source](https://tonkabayequity.com/news/corporate-technologies-acquires-numsp/)

**Sponsor-owned companies in this sector held 4+ years with no exit announced** (possible sponsor-to-sponsor deals):

- none in the collected announcements

**Supported by this evidence:** who announced which acquisitions, where, and when, and how long each sponsor says it has owned its platforms. **Needs further research:** companies nobody has bought yet (a company database or industry association lists), whether 'no exit announced' companies are really still held, deal sizes and prices, and the sponsors' fund life (an older fund sells sooner).

In [14]:
# What the announcements leave out: the honest size of the gaps (a missing price is never zero)
an.disclosure_gaps(since_2022[since_2022["focus_sector"].isin(so.FOCUS_SECTORS)])

,field,missing,share_missing
0,Transaction price,300,0.990099
1,Closing date,299,0.986799
2,Ownership stake,295,0.973597
3,Target location (state),115,0.379538
4,Target description,8,0.026403
5,Stated rationale,7,0.023102
6,Deal status (completed / pending),120,0.396040


**What the evidence supports:** who announced which acquisitions, where and when, and how long each sponsor says it
has owned its platforms. **What needs further research:** companies nobody has bought yet (a company database), whether
"no exit announced" companies are really still held, prices, and the sponsors' fund life. The collected announcements
are not a measure of the PE market.

## 7 · Extras: similar deals, the new-deal digest, exports

In [15]:
# More targets like Jones Bearing (target description and rationale in similar words)
an.similar_deals(deals, jones["deal_id"], k=5)[["similarity", "published_date", "publisher", "buyer", "target", "niche", "target_state"]]

,similarity,published_date,publisher,buyer,target,niche,target_state
0,0.249444,2022-11-30,Tonka Bay Equity Partners,Tonka Bay Equity Partners,"Standard Locknut, LLC",Precision components & machining,IN
1,0.195591,2018-11-01,ShoreView Industries,"Monroe Engineering Holdings, LLC","Magnus Mobility Systems, Inc",Precision components & machining,CA
2,0.155920,2016-01-06,Mason Wells,ProAmpac,"Coating Excellence International, LLC","Coatings, adhesives & chemicals",
3,0.155920,2021-11-03,CID Capital,Unknown,Seaga Manufacturing,Other: Industrial manufacturing,
4,0.138253,2024-01-29,Compass Group Equity Partners,Compass Group Equity Partners,Mearthane Products Corporation,Precision components & machining,


In [16]:
# Monitoring: everything published in the last 120 days of the data
latest = pd.to_datetime(deals["published_date"]).max()
an.new_since(deals, (latest - pd.Timedelta(days=120)).date().isoformat())[["published_date", "publisher", "buyer", "target", "transaction_type", "niche"]]

,published_date,publisher,buyer,target,transaction_type,niche
1047,2026-10-22,Fort Point Capital,Stord,AMS Fulfillment,Exit,Other: Value-added distribution
1045,2026-10-20,Fort Point Capital,NRG Controls,"Integrated Systems, Inc",Add-on acquisition,Building automation & energy services
1046,2026-10-20,Oridian Capital Partners,ACR,Ultra Green Packaging,Add-on acquisition,Packaging & plastics
1044,2026-10-14,Mason Wells,Mason Wells,"Canworks, Inc",Platform investment,
1043,2026-10-13,Goldner Hawn,Goldner Hawn,International Filtration Services,Platform investment,Filtration
1042,2026-10-07,Align Capital Partners,Advantage Investigations,NorStar Investigations,Add-on acquisition,Investigations & risk services
1041,2026-10-06,Gryphon Investors,Thermal Technology Distribution Solutions,Thermal Products,Add-on acquisition,Other: Value-added distribution
1039,2026-10-05,Gridiron Capital,Gridiron Capital,Weed Man,Platform investment,Broadband & network services
1040,2026-10-05,Compass Group Equity Partners,Unknown,Unknown,Unclear,
1037,2026-09-29,Compass Group Equity Partners,Avesi Partners,New Age Medical,Exit,


In [17]:
# Export: CSV / JSON / Markdown brief (saved sample outputs live in data/pe_deals/output/)
for p in ds.export("pe_deals_output"):
    print(p)

pe_deals_output/deals.csv
pe_deals_output/all_articles.csv
pe_deals_output/coverage.csv
pe_deals_output/failures.csv
pe_deals_output/deals.json
pe_deals_output/deal_brief.md
pe_deals_output/portfolio_ledger.csv
pe_deals_output/exit_watch.csv
pe_deals_output/consolidation_since_2022.csv


## Limitations

* Sixteen sponsors' self-published announcements: selective and incomplete, not the PE market.
* "No exit announced" is not proof a company is still held; entry dates found in passing are the sponsor's statement.
* Prices, closing dates and stakes are almost never disclosed; the fields show the gap instead of filling it.
* Pages with no publication date are excluded from the period and listed, never guessed.
* The rules fit these sites' press-release formulas; a new firm needs a source entry and a labelled check first.
* The reference labels were produced with AI assistance and still need a human spot-check (`data/pe_deals/validation/README.md`).
